# Experimentos Univariados v4 Cloud (Vertex AI)

**Tesis MEC** — 97 DGPs x T in {25,50,100,200} x R=500  
**Verificacion DGP:** cada experimento incluye seccion PASS/FAIL antes del Monte Carlo  
**Horizonte por T:** T=25->H=6 * T=50->H=18 * T=100,200->H=24  
**Metricas:** Bias, Varianza, RMSE, CRPS  
**Bloques:** Corto h=1-6 * Medio h=7-18 * Largo h=19-24  
**Logging:** dual stdout + `results/univariate_v3/run_YYYYMMDD_HHMMSS.log`  
**Resultados:** `results/univariate_v3/` — si existen se cargan sin re-simular

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import copy
import logging
import sys
import time
import traceback
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
import torch
from IPython.display import display

from statsmodels.tsa.stattools import adfuller, acf
from statsmodels.stats.diagnostic import acorr_ljungbox

from mectesis.dgp import (
    RandomWalk, SeasonalDGP,
    AR1ARCH, AR1GARCH,
    LocalLevelDGP, LocalTrendDGP, DampedTrendDGP, LocalLevelSeasonalDGP,
    ARpDGP, MAqDGP, ARMApqDGP, ARMApqWithTrendDGP,
    SETARDGp, LSTARDGp, ESTARDGp,
)
from mectesis.models import (
    ARIMAModel, ARIMAWithTrendModel, SARIMAModel,
    ARARCHModel, ARGARCHModel,
    ETSModel, ThetaModel, ChronosModel,
)
from mectesis.simulation import MonteCarloEngine

SEED    = 3649
H_BY_T  = {25: 6, 50: 18, 100: 24, 200: 24}
H_MAX   = 24
R_LIST  = [500]
T_LIST  = [25, 50, 100, 200]
RESULTS = Path("results/univariate_v3")
RESULTS.mkdir(parents=True, exist_ok=True)

# ── Logging dual: notebook + archivo .log ────────────────────────────────────
log_path = RESULTS / f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(message)s",
    datefmt="%H:%M:%S",
    handlers=[
        logging.FileHandler(log_path, encoding="utf-8"),
        logging.StreamHandler(sys.stdout),
    ],
)
log = logging.getLogger().info
log(f"Log en: {log_path}")

plt.rcParams.update({"figure.dpi": 110, "font.size": 10})
pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.max_columns", None)

device = "cuda" if torch.cuda.is_available() else "cpu"
log(f"Cargando Chronos-2 en {device} (puede tardar ~30 s la primera vez)...")
chronos = ChronosModel(device=device)
log("Chronos-2 listo.")

In [ ]:
# ─── Funciones auxiliares ────────────────────────────────────────────────────

def _cache_path(exp_id: str, T: int, R: int) -> Path:
    return RESULTS / f"exp_{exp_id.replace('.', '_')}_T{T}_R{R}.csv"


def _save_results(results: dict, path: Path):
    frames = []
    for mname, df in results.items():
        tmp = df.copy()
        tmp.insert(0, "model", mname)
        frames.append(tmp)
    pd.concat(frames, ignore_index=True).to_csv(path, index=False)


def _load_results(path: Path) -> dict:
    df = pd.read_csv(path)
    return {
        mname: grp.drop(columns="model").reset_index(drop=True)
        for mname, grp in df.groupby("model", sort=False)
    }


def run_exp(dgp, make_models_fn, dgp_params, exp_id,
            T_list=T_LIST, R_list=R_LIST, H_by_T=None, seed=SEED):
    if H_by_T is None:
        H_by_T = H_BY_T
    n_runs = len(T_list) * len(R_list)
    combos = ", ".join(
        f"(T={t}, H={H_by_T.get(t, H_MAX)}, R={r})"
        for t in T_list for r in R_list
    )
    log(f"Exp {exp_id}: {n_runs} ejecucion(es) -> {combos}")
    all_results = {}
    for T in T_list:
        h = H_by_T.get(T, H_MAX)
        for R in R_list:
            cache = _cache_path(exp_id, T, R)
            if cache.exists():
                log(f"  T={T} H={h}, R={R}: cargando {cache.name}")
                all_results[(T, R)] = _load_results(cache)
                continue
            log(f"  T={T} H={h}, R={R}: simulando...")
            dgp.rng = np.random.default_rng(seed)
            models = make_models_fn(T)
            engine = MonteCarloEngine(dgp, models, seed=seed)
            t0 = time.time()
            results = engine.run_monte_carlo(R, T, h, dgp_params, verbose=False)
            log(f"  T={T} H={h}, R={R}: OK ({time.time()-t0:.0f}s)")
            _save_results(results, cache)
            all_results[(T, R)] = results
    return all_results


# ─── Funciones v3 ────────────────────────────────────────────────────────────

BLOCK_DEFS = [("C", 1, 6), ("M", 7, 18), ("L", 19, 24)]
METRICS_V3  = ["bias", "variance", "rmse", "crps"]


def compute_blocks_v3(results_TR: dict) -> dict:
    out = {}
    for mname, df in results_TR.items():
        df_h = df[df["horizon"] != "avg_all"].copy()
        df_h["horizon"] = pd.to_numeric(df_h["horizon"], errors="coerce")
        blks = {}
        for blk, h1, h2 in BLOCK_DEFS:
            mask = (df_h["horizon"] >= h1) & (df_h["horizon"] <= h2)
            blks[blk] = df_h[mask].mean(numeric_only=True)
        out[mname] = blks
    return out


def build_grid_table(all_results: dict, classical_name: str,
                     chronos_name: str = "Chronos-2"):
    rows = []
    for (T, R), res_TR in sorted(all_results.items()):
        blk_data = compute_blocks_v3(res_TR)
        cl_blks  = blk_data.get(classical_name, {})
        ch_blks  = blk_data.get(chronos_name, {})

        for mname, blks in blk_data.items():
            row = {"T": T, "Modelo": mname}
            for blk, h1, h2 in BLOCK_DEFS:
                s = blks.get(blk, pd.Series(dtype=float))
                for m in METRICS_V3:
                    row[f"{m}_{blk}"] = (
                        round(float(s[m]), 4)
                        if m in s.index and pd.notna(s[m]) else np.nan
                    )
                cl_s = cl_blks.get(blk, pd.Series(dtype=float))
                ch_s = ch_blks.get(blk, pd.Series(dtype=float))
                for m in ["rmse", "crps"]:
                    cv = float(cl_s[m]) if m in cl_s.index and pd.notna(cl_s[m]) else np.nan
                    hv = float(ch_s[m]) if m in ch_s.index and pd.notna(ch_s[m]) else np.nan
                    if np.isnan(cv) or np.isnan(hv):
                        row[f"best_{m}_{blk}"] = np.nan
                    else:
                        row[f"best_{m}_{blk}"] = "C" if cv <= hv else "T"
            rows.append(row)

    df_out = pd.DataFrame(rows).set_index(["T", "Modelo"])
    display(df_out.style.format(precision=4, na_rep="—"))


def plot_simulation_v3(dgp, models, dgp_params, title="", T_vis=100, seed=SEED):
    H_vis   = H_BY_T.get(T_vis, H_MAX)
    old_rng = dgp.rng
    dgp.rng = np.random.default_rng(seed + 99991)
    y       = dgp.simulate(T=T_vis, **dgp_params)
    dgp.rng = old_rng

    split   = T_vis - H_vis
    y_train = y[:split]
    y_test  = y[split:]
    t_train = np.arange(split)
    t_test  = np.arange(split, T_vis)

    fig, ax = plt.subplots(figsize=(10, 3.5))
    ax.plot(t_train, y_train, color="gray", lw=1.5, label="Historico")
    ax.axvline(split - 0.5, color="black", ls="--", lw=1, alpha=0.6)
    ax.plot(t_test, y_test, color="black", lw=1.5, marker="o", ms=3, label="Observado")

    palette = ["steelblue", "darkorange", "seagreen", "purple"]
    for i, model in enumerate(models):
        try:
            model.fit(y_train)
            fcst = model.forecast(H_vis)
            c = palette[i % len(palette)]
            ax.plot(t_test, fcst, color=c, lw=1.5, ls="--", marker="s", ms=3, label=model.name)
            if getattr(model, "supports_intervals", False):
                lo, hi = model.forecast_intervals(H_vis, level=0.80)
                ax.fill_between(t_test, lo, hi, color=c, alpha=0.15)
        except Exception as e:
            log(f"  [plot] {model.name} fallo: {e}")

    ax.set(title=title, xlabel="t", ylabel="y")
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()


# ─── Funciones de verificacion DGP ───────────────────────────────────────────

def verify_dgp(label, dgp, dgp_params, classical_model, checks):
    log(f"{'─'*60}")
    log(f"VERIFICACION DGP: {label}")
    log(f"{'─'*60}")
    dgp_copy = copy.deepcopy(dgp)
    dgp_copy.rng = np.random.default_rng(7777)
    try:
        y_long = dgp_copy.simulate(T=1000, **dgp_params)
    except Exception as e:
        log(f"  [FAIL] simulate() lanzo excepcion: {e}")
        return
    n_fail = 0
    for check_name, check_fn in checks:
        try:
            ok, msg = check_fn(y_long, dgp, dgp_params, classical_model)
        except Exception as e:
            ok, msg = False, f"excepcion inesperada: {e}"
        tag = "PASS" if ok else "FAIL"
        log(f"  [{tag}] {check_name}: {msg}")
        if not ok:
            n_fail += 1
    if n_fail == 0:
        log("  -> TODAS LAS VERIFICACIONES PASARON")
    else:
        log(f"  -> {n_fail} FALLO(S)")


def chk_stationary(y, *_):
    pval = adfuller(y, autolag="AIC")[1]
    return pval < 0.05, f"ADF p={pval:.4f} (umbral 0.05)"

def chk_nonstationary(y, *_):
    pval = adfuller(y, autolag="AIC")[1]
    return pval > 0.10, f"ADF p={pval:.4f} (se espera >0.10)"

def chk_zero_mean(y, *_):
    mu = y.mean(); sigma = y.std(); T = len(y)
    tol = 3.0 * sigma / np.sqrt(T)
    return abs(mu) < tol, f"media={mu:.4f}, tol=+/-{tol:.4f}"

def chk_acf_lag1(y, dgp, dgp_params, *_):
    props = dgp.get_theoretical_properties(**dgp_params) if dgp_params else dgp.get_theoretical_properties()
    phi1 = props.get("phis", [None])[0] if props.get("phis") else None
    if phi1 is None:
        return True, "no aplica (sin phis)"
    acf_vals = acf(y, nlags=1, fft=True)
    emp = acf_vals[1]
    ok = abs(emp - phi1) < 0.15
    return ok, f"ACF[1]={emp:.4f}, phi1_DGP={phi1:.4f}, dif={abs(emp-phi1):.4f}"

def chk_ma_cutoff(y, dgp, dgp_params, *_):
    props = dgp.get_theoretical_properties(**dgp_params) if dgp_params else dgp.get_theoretical_properties()
    thetas = props.get("thetas", [])
    q = len(thetas)
    if q == 0:
        return True, "no aplica (sin thetas)"
    T = len(y)
    thr = 2.0 / np.sqrt(T)
    acf_vals = acf(y, nlags=q+2, fft=True)
    val_after = abs(acf_vals[q+1])
    ok = val_after < thr
    return ok, f"|ACF[{q+1}]|={val_after:.4f}, umbral=2/sqrt(T)={thr:.4f}"

def chk_trend_slope(y, dgp, dgp_params, *_):
    delta = dgp_params.get("delta", None)
    if delta is None:
        return True, "no aplica (sin delta)"
    T = len(y)
    slope = np.polyfit(np.arange(T), y, 1)[0]
    ok = abs(slope - delta) < 0.015
    return ok, f"pendiente_OLS={slope:.5f}, delta_DGP={delta:.5f}, dif={abs(slope-delta):.5f}"

def chk_arch_effects(y, *_):
    y2 = y ** 2
    lb = acorr_ljungbox(y2, lags=[10], return_df=True)
    pval = float(lb["lb_pvalue"].iloc[0])
    return pval < 0.05, f"LB(10) sobre y^2: p={pval:.4f} (se espera <0.05)"

def chk_rw_variance_growth(y, *_):
    T = len(y)
    v1 = np.var(y[:T//4])
    v2 = np.var(y[T//2:])
    ok = v2 > 2.0 * v1
    return ok, f"var(primera cuarta)={v1:.4f}, var(segunda mitad)={v2:.4f}"

def chk_seasonal_acf(y, dgp, dgp_params, *_):
    s = dgp_params.get("s", 4)
    T = len(y)
    thr = 2.0 / np.sqrt(T)
    acf_vals = acf(y, nlags=s+1, fft=True)
    val_s = abs(acf_vals[s])
    ok = val_s > thr
    return ok, f"|ACF[{s}]|={val_s:.4f}, umbral=2/sqrt(T)={thr:.4f}"

def chk_fit_classical(y, dgp, dgp_params, model):
    try:
        m = copy.deepcopy(model)
        m.fit(y[:800])
        fc = m.forecast(horizon=6)
        ok = len(fc) == 6 and not np.any(np.isnan(fc))
        return ok, f"fit+forecast OK, primeros valores: {fc[:3].round(4)}"
    except Exception as e:
        return False, str(e)

def chk_arma_aic(y, dgp, dgp_params, model):
    from statsmodels.tsa.arima.model import ARIMA as StatsARIMA
    props = dgp.get_theoretical_properties(**dgp_params) if dgp_params else dgp.get_theoretical_properties()
    p = len(props.get("phis", []))
    q = len(props.get("thetas", []))
    try:
        aic_arma = StatsARIMA(y, order=(p, 0, q)).fit(disp=False).aic
        aic_ar   = StatsARIMA(y, order=(p, 0, 0)).fit(disp=False).aic if p > 0 else np.inf
        aic_ma   = StatsARIMA(y, order=(0, 0, q)).fit(disp=False).aic if q > 0 else np.inf
        baseline = min(aic_ar, aic_ma)
        ok = aic_arma < baseline
        return ok, f"AIC_ARMA={aic_arma:.1f}, AIC_baseline={baseline:.1f}"
    except Exception as e:
        return False, str(e)

def chk_theta_fit(y, dgp, dgp_params, model):
    try:
        m = copy.deepcopy(model)
        m.fit(y[:800])
        fc = m.forecast(horizon=6)
        return True, f"OK, primeros valores: {fc[:3].round(4)}"
    except Exception as e:
        return False, str(e)

# ── grupos de checks por tipo ─────────────────────────────────────────────────

CHECKS_AR = [
    ("Estacionariedad (ADF)", chk_stationary),
    ("Media ~ 0",             chk_zero_mean),
    ("ACF[1] ~ phi1",         chk_acf_lag1),
    ("Ajuste modelo clasico", chk_fit_classical),
]
CHECKS_AR_TREND = [
    ("Pendiente OLS ~ delta", chk_trend_slope),
    ("Ajuste modelo clasico", chk_fit_classical),
]
CHECKS_MA = [
    ("Estacionariedad (ADF)", chk_stationary),
    ("Media ~ 0",             chk_zero_mean),
    ("Corte ACF en lag q",    chk_ma_cutoff),
    ("Ajuste modelo clasico", chk_fit_classical),
]
CHECKS_MA_TREND = [
    ("Pendiente OLS ~ delta", chk_trend_slope),
    ("Ajuste modelo clasico", chk_fit_classical),
]
CHECKS_ARMA = [
    ("Estacionariedad (ADF)", chk_stationary),
    ("AIC ARMA < baseline",   chk_arma_aic),
    ("Ajuste modelo clasico", chk_fit_classical),
]
CHECKS_RW = [
    ("No estacionariedad",    chk_nonstationary),
    ("Varianza crece con t",  chk_rw_variance_growth),
    ("Ajuste modelo clasico", chk_fit_classical),
]
CHECKS_ARCH = [
    ("Estacionariedad media", chk_stationary),
    ("Efectos ARCH (LB y^2)", chk_arch_effects),
    ("Ajuste modelo clasico", chk_fit_classical),
]
CHECKS_SAR = [
    ("Estacionariedad (ADF)", chk_stationary),
    ("ACF estacional signif.", chk_seasonal_acf),
    ("Ajuste modelo clasico", chk_fit_classical),
]
CHECKS_ETS = [
    ("No estacionariedad",    chk_nonstationary),
    ("Ajuste modelo clasico", chk_fit_classical),
]
CHECKS_THETA = [
    ("Ajuste Theta",          chk_theta_fit),
]
CHECKS_NONLINEAR = [
    ("Estacionariedad (ADF)", chk_stationary),
    ("Ajuste modelo clasico", chk_fit_classical),
]

---
## Bloque A — ARMA sin tendencia (24 experimentos)

DGPs: AR(1-4) * MA(1-4) * ARMA(1,1) * ARMA(2,2) * ARMA(1,4) * ARMA(4,1)  
Modelo clasico: ARIMA(p,0,q) correctamente especificado vs Chronos-2

In [ ]:
try:
    # A.1 -- AR(1) rho=0.30
    cl  = ARIMAModel((1, 0, 0))
    dgp = ARpDGP(phis=[0.3], sigma=1.0, seed=SEED)
    verify_dgp("A.1 -- AR(1) rho=0.30", dgp, {}, cl, CHECKS_AR)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="A.1")
    log("\n" + "="*60 + "\nA.1 -- AR(1) rho=0.30\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(1, 0, 0)")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="A.1 -- AR(1) rho=0.30")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA A.1 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # A.2 -- AR(1) rho=0.90
    cl  = ARIMAModel((1, 0, 0))
    dgp = ARpDGP(phis=[0.9], sigma=1.0, seed=SEED)
    verify_dgp("A.2 -- AR(1) rho=0.90", dgp, {}, cl, CHECKS_AR)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="A.2")
    log("\n" + "="*60 + "\nA.2 -- AR(1) rho=0.90\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(1, 0, 0)")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="A.2 -- AR(1) rho=0.90")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA A.2 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # A.3 -- AR(2) rho=0.30
    cl  = ARIMAModel((2, 0, 0))
    dgp = ARpDGP(phis=[0.3, 0.1], sigma=1.0, seed=SEED)
    verify_dgp("A.3 -- AR(2) rho=0.30", dgp, {}, cl, CHECKS_AR)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="A.3")
    log("\n" + "="*60 + "\nA.3 -- AR(2) rho=0.30\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(2, 0, 0)")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="A.3 -- AR(2) rho=0.30")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA A.3 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # A.4 -- AR(2) rho=0.90
    cl  = ARIMAModel((2, 0, 0))
    dgp = ARpDGP(phis=[0.9, -0.2], sigma=1.0, seed=SEED)
    verify_dgp("A.4 -- AR(2) rho=0.90", dgp, {}, cl, CHECKS_AR)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="A.4")
    log("\n" + "="*60 + "\nA.4 -- AR(2) rho=0.90\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(2, 0, 0)")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="A.4 -- AR(2) rho=0.90")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA A.4 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # A.5 -- AR(3) rho=0.30
    cl  = ARIMAModel((3, 0, 0))
    dgp = ARpDGP(phis=[0.3, 0.1, 0.05], sigma=1.0, seed=SEED)
    verify_dgp("A.5 -- AR(3) rho=0.30", dgp, {}, cl, CHECKS_AR)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="A.5")
    log("\n" + "="*60 + "\nA.5 -- AR(3) rho=0.30\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(3, 0, 0)")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="A.5 -- AR(3) rho=0.30")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA A.5 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # A.6 -- AR(3) rho=0.90
    cl  = ARIMAModel((3, 0, 0))
    dgp = ARpDGP(phis=[0.9, -0.2, 0.1], sigma=1.0, seed=SEED)
    verify_dgp("A.6 -- AR(3) rho=0.90", dgp, {}, cl, CHECKS_AR)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="A.6")
    log("\n" + "="*60 + "\nA.6 -- AR(3) rho=0.90\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(3, 0, 0)")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="A.6 -- AR(3) rho=0.90")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA A.6 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # A.7 -- AR(4) rho=0.30
    cl  = ARIMAModel((4, 0, 0))
    dgp = ARpDGP(phis=[0.3, 0.1, 0.05, 0.02], sigma=1.0, seed=SEED)
    verify_dgp("A.7 -- AR(4) rho=0.30", dgp, {}, cl, CHECKS_AR)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="A.7")
    log("\n" + "="*60 + "\nA.7 -- AR(4) rho=0.30\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(4, 0, 0)")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="A.7 -- AR(4) rho=0.30")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA A.7 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # A.8 -- AR(4) rho=0.90
    cl  = ARIMAModel((4, 0, 0))
    dgp = ARpDGP(phis=[0.9, -0.2, 0.1, -0.05], sigma=1.0, seed=SEED)
    verify_dgp("A.8 -- AR(4) rho=0.90", dgp, {}, cl, CHECKS_AR)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="A.8")
    log("\n" + "="*60 + "\nA.8 -- AR(4) rho=0.90\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(4, 0, 0)")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="A.8 -- AR(4) rho=0.90")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA A.8 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # A.9 -- MA(1) theta=0.30
    cl  = ARIMAModel((0, 0, 1))
    dgp = MAqDGP(thetas=[0.3], sigma=1.0, seed=SEED)
    verify_dgp("A.9 -- MA(1) theta=0.30", dgp, {}, cl, CHECKS_MA)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="A.9")
    log("\n" + "="*60 + "\nA.9 -- MA(1) theta=0.30\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(0, 0, 1)")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="A.9 -- MA(1) theta=0.30")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA A.9 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # A.10 -- MA(1) theta=0.90
    cl  = ARIMAModel((0, 0, 1))
    dgp = MAqDGP(thetas=[0.9], sigma=1.0, seed=SEED)
    verify_dgp("A.10 -- MA(1) theta=0.90", dgp, {}, cl, CHECKS_MA)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="A.10")
    log("\n" + "="*60 + "\nA.10 -- MA(1) theta=0.90\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(0, 0, 1)")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="A.10 -- MA(1) theta=0.90")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA A.10 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # A.11 -- MA(2) theta=0.30
    cl  = ARIMAModel((0, 0, 2))
    dgp = MAqDGP(thetas=[0.3, 0.1], sigma=1.0, seed=SEED)
    verify_dgp("A.11 -- MA(2) theta=0.30", dgp, {}, cl, CHECKS_MA)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="A.11")
    log("\n" + "="*60 + "\nA.11 -- MA(2) theta=0.30\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(0, 0, 2)")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="A.11 -- MA(2) theta=0.30")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA A.11 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # A.12 -- MA(2) theta=0.90
    cl  = ARIMAModel((0, 0, 2))
    dgp = MAqDGP(thetas=[0.9, 0.1], sigma=1.0, seed=SEED)
    verify_dgp("A.12 -- MA(2) theta=0.90", dgp, {}, cl, CHECKS_MA)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="A.12")
    log("\n" + "="*60 + "\nA.12 -- MA(2) theta=0.90\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(0, 0, 2)")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="A.12 -- MA(2) theta=0.90")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA A.12 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # A.13 -- MA(3) theta=0.30
    cl  = ARIMAModel((0, 0, 3))
    dgp = MAqDGP(thetas=[0.3, 0.1, -0.05], sigma=1.0, seed=SEED)
    verify_dgp("A.13 -- MA(3) theta=0.30", dgp, {}, cl, CHECKS_MA)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="A.13")
    log("\n" + "="*60 + "\nA.13 -- MA(3) theta=0.30\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(0, 0, 3)")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="A.13 -- MA(3) theta=0.30")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA A.13 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # A.14 -- MA(3) theta=0.90
    cl  = ARIMAModel((0, 0, 3))
    dgp = MAqDGP(thetas=[0.9, 0.1, -0.05], sigma=1.0, seed=SEED)
    verify_dgp("A.14 -- MA(3) theta=0.90", dgp, {}, cl, CHECKS_MA)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="A.14")
    log("\n" + "="*60 + "\nA.14 -- MA(3) theta=0.90\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(0, 0, 3)")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="A.14 -- MA(3) theta=0.90")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA A.14 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # A.15 -- MA(4) theta=0.30
    cl  = ARIMAModel((0, 0, 4))
    dgp = MAqDGP(thetas=[0.3, 0.1, -0.05, 0.02], sigma=1.0, seed=SEED)
    verify_dgp("A.15 -- MA(4) theta=0.30", dgp, {}, cl, CHECKS_MA)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="A.15")
    log("\n" + "="*60 + "\nA.15 -- MA(4) theta=0.30\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(0, 0, 4)")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="A.15 -- MA(4) theta=0.30")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA A.15 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # A.16 -- MA(4) theta=0.90
    cl  = ARIMAModel((0, 0, 4))
    dgp = MAqDGP(thetas=[0.9, 0.1, -0.05, 0.02], sigma=1.0, seed=SEED)
    verify_dgp("A.16 -- MA(4) theta=0.90", dgp, {}, cl, CHECKS_MA)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="A.16")
    log("\n" + "="*60 + "\nA.16 -- MA(4) theta=0.90\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(0, 0, 4)")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="A.16 -- MA(4) theta=0.90")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA A.16 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # A.17 -- ARMA(1,1) rho=0.30
    cl  = ARIMAModel((1, 0, 1))
    dgp = ARMApqDGP(phis=[0.3], thetas=[0.1], sigma=1.0, seed=SEED)
    verify_dgp("A.17 -- ARMA(1,1) rho=0.30", dgp, {}, cl, CHECKS_ARMA)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="A.17")
    log("\n" + "="*60 + "\nA.17 -- ARMA(1,1) rho=0.30\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(1, 0, 1)")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="A.17 -- ARMA(1,1) rho=0.30")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA A.17 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # A.18 -- ARMA(1,1) rho=0.90
    cl  = ARIMAModel((1, 0, 1))
    dgp = ARMApqDGP(phis=[0.9], thetas=[0.3], sigma=1.0, seed=SEED)
    verify_dgp("A.18 -- ARMA(1,1) rho=0.90", dgp, {}, cl, CHECKS_ARMA)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="A.18")
    log("\n" + "="*60 + "\nA.18 -- ARMA(1,1) rho=0.90\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(1, 0, 1)")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="A.18 -- ARMA(1,1) rho=0.90")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA A.18 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # A.19 -- ARMA(2,2) rho=0.30
    cl  = ARIMAModel((2, 0, 2))
    dgp = ARMApqDGP(phis=[0.3, 0.1], thetas=[0.1, 0.05], sigma=1.0, seed=SEED)
    verify_dgp("A.19 -- ARMA(2,2) rho=0.30", dgp, {}, cl, CHECKS_ARMA)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="A.19")
    log("\n" + "="*60 + "\nA.19 -- ARMA(2,2) rho=0.30\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(2, 0, 2)")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="A.19 -- ARMA(2,2) rho=0.30")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA A.19 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # A.20 -- ARMA(2,2) rho=0.90
    cl  = ARIMAModel((2, 0, 2))
    dgp = ARMApqDGP(phis=[0.9, -0.2], thetas=[0.3, -0.1], sigma=1.0, seed=SEED)
    verify_dgp("A.20 -- ARMA(2,2) rho=0.90", dgp, {}, cl, CHECKS_ARMA)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="A.20")
    log("\n" + "="*60 + "\nA.20 -- ARMA(2,2) rho=0.90\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(2, 0, 2)")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="A.20 -- ARMA(2,2) rho=0.90")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA A.20 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # A.21 -- ARMA(1,4) rho=0.30
    cl  = ARIMAModel((1, 0, 4))
    dgp = ARMApqDGP(phis=[0.3], thetas=[0.1, 0.05, -0.03, 0.01], sigma=1.0, seed=SEED)
    verify_dgp("A.21 -- ARMA(1,4) rho=0.30", dgp, {}, cl, CHECKS_ARMA)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="A.21")
    log("\n" + "="*60 + "\nA.21 -- ARMA(1,4) rho=0.30\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(1, 0, 4)")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="A.21 -- ARMA(1,4) rho=0.30")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA A.21 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # A.22 -- ARMA(1,4) rho=0.90
    cl  = ARIMAModel((1, 0, 4))
    dgp = ARMApqDGP(phis=[0.9], thetas=[0.3, -0.1, 0.05, -0.02], sigma=1.0, seed=SEED)
    verify_dgp("A.22 -- ARMA(1,4) rho=0.90", dgp, {}, cl, CHECKS_ARMA)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="A.22")
    log("\n" + "="*60 + "\nA.22 -- ARMA(1,4) rho=0.90\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(1, 0, 4)")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="A.22 -- ARMA(1,4) rho=0.90")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA A.22 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # A.23 -- ARMA(4,1) rho=0.30
    cl  = ARIMAModel((4, 0, 1))
    dgp = ARMApqDGP(phis=[0.3, 0.1, 0.05, 0.02], thetas=[0.1], sigma=1.0, seed=SEED)
    verify_dgp("A.23 -- ARMA(4,1) rho=0.30", dgp, {}, cl, CHECKS_ARMA)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="A.23")
    log("\n" + "="*60 + "\nA.23 -- ARMA(4,1) rho=0.30\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(4, 0, 1)")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="A.23 -- ARMA(4,1) rho=0.30")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA A.23 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # A.24 -- ARMA(4,1) rho=0.90
    cl  = ARIMAModel((4, 0, 1))
    dgp = ARMApqDGP(phis=[0.9, -0.2, 0.1, -0.05], thetas=[0.3], sigma=1.0, seed=SEED)
    verify_dgp("A.24 -- ARMA(4,1) rho=0.90", dgp, {}, cl, CHECKS_ARMA)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="A.24")
    log("\n" + "="*60 + "\nA.24 -- ARMA(4,1) rho=0.90\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(4, 0, 1)")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="A.24 -- ARMA(4,1) rho=0.90")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA A.24 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

---
## Bloque B — ARMA con tendencia deterministica (48 experimentos)

DGP: `Y_t = alpha + delta*t + ARMA_t`  
Tendencia leve: delta=0.02 (B.1-B.24) | Tendencia fuerte: delta=0.10 (B.25-B.48)  
Modelo clasico: ARIMA(p,0,q)+trend (trend='ct')

In [ ]:
try:
    # B.1 -- AR(1) rho=0.30  [tendencia leve delta=0.02]
    cl  = ARIMAWithTrendModel((1, 0, 0), trend="ct")
    dgp = ARMApqWithTrendDGP(phis=[0.3], thetas=[], delta=0.02, sigma=1.0, seed=SEED)
    verify_dgp("B.1 -- AR(1) rho=0.30 (delta=0.02)", dgp, {}, cl, CHECKS_AR_TREND)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.1")
    log("\n" + "="*60 + "\nB.1 -- AR(1) rho=0.30 (delta=0.02)\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(1, 0, 0)+trend")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="B.1 -- AR(1) rho=0.30 (delta=0.02)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA B.1 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # B.2 -- AR(1) rho=0.90  [tendencia leve delta=0.02]
    cl  = ARIMAWithTrendModel((1, 0, 0), trend="ct")
    dgp = ARMApqWithTrendDGP(phis=[0.9], thetas=[], delta=0.02, sigma=1.0, seed=SEED)
    verify_dgp("B.2 -- AR(1) rho=0.90 (delta=0.02)", dgp, {}, cl, CHECKS_AR_TREND)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.2")
    log("\n" + "="*60 + "\nB.2 -- AR(1) rho=0.90 (delta=0.02)\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(1, 0, 0)+trend")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="B.2 -- AR(1) rho=0.90 (delta=0.02)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA B.2 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # B.3 -- AR(2) rho=0.30  [tendencia leve delta=0.02]
    cl  = ARIMAWithTrendModel((2, 0, 0), trend="ct")
    dgp = ARMApqWithTrendDGP(phis=[0.3, 0.1], thetas=[], delta=0.02, sigma=1.0, seed=SEED)
    verify_dgp("B.3 -- AR(2) rho=0.30 (delta=0.02)", dgp, {}, cl, CHECKS_AR_TREND)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.3")
    log("\n" + "="*60 + "\nB.3 -- AR(2) rho=0.30 (delta=0.02)\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(2, 0, 0)+trend")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="B.3 -- AR(2) rho=0.30 (delta=0.02)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA B.3 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # B.4 -- AR(2) rho=0.90  [tendencia leve delta=0.02]
    cl  = ARIMAWithTrendModel((2, 0, 0), trend="ct")
    dgp = ARMApqWithTrendDGP(phis=[0.9, -0.2], thetas=[], delta=0.02, sigma=1.0, seed=SEED)
    verify_dgp("B.4 -- AR(2) rho=0.90 (delta=0.02)", dgp, {}, cl, CHECKS_AR_TREND)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.4")
    log("\n" + "="*60 + "\nB.4 -- AR(2) rho=0.90 (delta=0.02)\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(2, 0, 0)+trend")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="B.4 -- AR(2) rho=0.90 (delta=0.02)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA B.4 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # B.5 -- AR(3) rho=0.30  [tendencia leve delta=0.02]
    cl  = ARIMAWithTrendModel((3, 0, 0), trend="ct")
    dgp = ARMApqWithTrendDGP(phis=[0.3, 0.1, 0.05], thetas=[], delta=0.02, sigma=1.0, seed=SEED)
    verify_dgp("B.5 -- AR(3) rho=0.30 (delta=0.02)", dgp, {}, cl, CHECKS_AR_TREND)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.5")
    log("\n" + "="*60 + "\nB.5 -- AR(3) rho=0.30 (delta=0.02)\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(3, 0, 0)+trend")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="B.5 -- AR(3) rho=0.30 (delta=0.02)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA B.5 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # B.6 -- AR(3) rho=0.90  [tendencia leve delta=0.02]
    cl  = ARIMAWithTrendModel((3, 0, 0), trend="ct")
    dgp = ARMApqWithTrendDGP(phis=[0.9, -0.2, 0.1], thetas=[], delta=0.02, sigma=1.0, seed=SEED)
    verify_dgp("B.6 -- AR(3) rho=0.90 (delta=0.02)", dgp, {}, cl, CHECKS_AR_TREND)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.6")
    log("\n" + "="*60 + "\nB.6 -- AR(3) rho=0.90 (delta=0.02)\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(3, 0, 0)+trend")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="B.6 -- AR(3) rho=0.90 (delta=0.02)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA B.6 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # B.7 -- AR(4) rho=0.30  [tendencia leve delta=0.02]
    cl  = ARIMAWithTrendModel((4, 0, 0), trend="ct")
    dgp = ARMApqWithTrendDGP(phis=[0.3, 0.1, 0.05, 0.02], thetas=[], delta=0.02, sigma=1.0, seed=SEED)
    verify_dgp("B.7 -- AR(4) rho=0.30 (delta=0.02)", dgp, {}, cl, CHECKS_AR_TREND)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.7")
    log("\n" + "="*60 + "\nB.7 -- AR(4) rho=0.30 (delta=0.02)\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(4, 0, 0)+trend")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="B.7 -- AR(4) rho=0.30 (delta=0.02)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA B.7 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # B.8 -- AR(4) rho=0.90  [tendencia leve delta=0.02]
    cl  = ARIMAWithTrendModel((4, 0, 0), trend="ct")
    dgp = ARMApqWithTrendDGP(phis=[0.9, -0.2, 0.1, -0.05], thetas=[], delta=0.02, sigma=1.0, seed=SEED)
    verify_dgp("B.8 -- AR(4) rho=0.90 (delta=0.02)", dgp, {}, cl, CHECKS_AR_TREND)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.8")
    log("\n" + "="*60 + "\nB.8 -- AR(4) rho=0.90 (delta=0.02)\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(4, 0, 0)+trend")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="B.8 -- AR(4) rho=0.90 (delta=0.02)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA B.8 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # B.9 -- MA(1) theta=0.30  [tendencia leve delta=0.02]
    cl  = ARIMAWithTrendModel((0, 0, 1), trend="ct")
    dgp = ARMApqWithTrendDGP(phis=[], thetas=[0.3], delta=0.02, sigma=1.0, seed=SEED)
    verify_dgp("B.9 -- MA(1) theta=0.30 (delta=0.02)", dgp, {}, cl, CHECKS_MA_TREND)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.9")
    log("\n" + "="*60 + "\nB.9 -- MA(1) theta=0.30 (delta=0.02)\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(0, 0, 1)+trend")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="B.9 -- MA(1) theta=0.30 (delta=0.02)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA B.9 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # B.10 -- MA(1) theta=0.90  [tendencia leve delta=0.02]
    cl  = ARIMAWithTrendModel((0, 0, 1), trend="ct")
    dgp = ARMApqWithTrendDGP(phis=[], thetas=[0.9], delta=0.02, sigma=1.0, seed=SEED)
    verify_dgp("B.10 -- MA(1) theta=0.90 (delta=0.02)", dgp, {}, cl, CHECKS_MA_TREND)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.10")
    log("\n" + "="*60 + "\nB.10 -- MA(1) theta=0.90 (delta=0.02)\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(0, 0, 1)+trend")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="B.10 -- MA(1) theta=0.90 (delta=0.02)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA B.10 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # B.11 -- MA(2) theta=0.30  [tendencia leve delta=0.02]
    cl  = ARIMAWithTrendModel((0, 0, 2), trend="ct")
    dgp = ARMApqWithTrendDGP(phis=[], thetas=[0.3, 0.1], delta=0.02, sigma=1.0, seed=SEED)
    verify_dgp("B.11 -- MA(2) theta=0.30 (delta=0.02)", dgp, {}, cl, CHECKS_MA_TREND)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.11")
    log("\n" + "="*60 + "\nB.11 -- MA(2) theta=0.30 (delta=0.02)\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(0, 0, 2)+trend")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="B.11 -- MA(2) theta=0.30 (delta=0.02)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA B.11 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # B.12 -- MA(2) theta=0.90  [tendencia leve delta=0.02]
    cl  = ARIMAWithTrendModel((0, 0, 2), trend="ct")
    dgp = ARMApqWithTrendDGP(phis=[], thetas=[0.9, 0.1], delta=0.02, sigma=1.0, seed=SEED)
    verify_dgp("B.12 -- MA(2) theta=0.90 (delta=0.02)", dgp, {}, cl, CHECKS_MA_TREND)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.12")
    log("\n" + "="*60 + "\nB.12 -- MA(2) theta=0.90 (delta=0.02)\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(0, 0, 2)+trend")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="B.12 -- MA(2) theta=0.90 (delta=0.02)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA B.12 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # B.13 -- MA(3) theta=0.30  [tendencia leve delta=0.02]
    cl  = ARIMAWithTrendModel((0, 0, 3), trend="ct")
    dgp = ARMApqWithTrendDGP(phis=[], thetas=[0.3, 0.1, -0.05], delta=0.02, sigma=1.0, seed=SEED)
    verify_dgp("B.13 -- MA(3) theta=0.30 (delta=0.02)", dgp, {}, cl, CHECKS_MA_TREND)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.13")
    log("\n" + "="*60 + "\nB.13 -- MA(3) theta=0.30 (delta=0.02)\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(0, 0, 3)+trend")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="B.13 -- MA(3) theta=0.30 (delta=0.02)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA B.13 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # B.14 -- MA(3) theta=0.90  [tendencia leve delta=0.02]
    cl  = ARIMAWithTrendModel((0, 0, 3), trend="ct")
    dgp = ARMApqWithTrendDGP(phis=[], thetas=[0.9, 0.1, -0.05], delta=0.02, sigma=1.0, seed=SEED)
    verify_dgp("B.14 -- MA(3) theta=0.90 (delta=0.02)", dgp, {}, cl, CHECKS_MA_TREND)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.14")
    log("\n" + "="*60 + "\nB.14 -- MA(3) theta=0.90 (delta=0.02)\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(0, 0, 3)+trend")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="B.14 -- MA(3) theta=0.90 (delta=0.02)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA B.14 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # B.15 -- MA(4) theta=0.30  [tendencia leve delta=0.02]
    cl  = ARIMAWithTrendModel((0, 0, 4), trend="ct")
    dgp = ARMApqWithTrendDGP(phis=[], thetas=[0.3, 0.1, -0.05, 0.02], delta=0.02, sigma=1.0, seed=SEED)
    verify_dgp("B.15 -- MA(4) theta=0.30 (delta=0.02)", dgp, {}, cl, CHECKS_MA_TREND)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.15")
    log("\n" + "="*60 + "\nB.15 -- MA(4) theta=0.30 (delta=0.02)\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(0, 0, 4)+trend")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="B.15 -- MA(4) theta=0.30 (delta=0.02)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA B.15 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # B.16 -- MA(4) theta=0.90  [tendencia leve delta=0.02]
    cl  = ARIMAWithTrendModel((0, 0, 4), trend="ct")
    dgp = ARMApqWithTrendDGP(phis=[], thetas=[0.9, 0.1, -0.05, 0.02], delta=0.02, sigma=1.0, seed=SEED)
    verify_dgp("B.16 -- MA(4) theta=0.90 (delta=0.02)", dgp, {}, cl, CHECKS_MA_TREND)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.16")
    log("\n" + "="*60 + "\nB.16 -- MA(4) theta=0.90 (delta=0.02)\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(0, 0, 4)+trend")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="B.16 -- MA(4) theta=0.90 (delta=0.02)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA B.16 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # B.17 -- ARMA(1,1) rho=0.30  [tendencia leve delta=0.02]
    cl  = ARIMAWithTrendModel((1, 0, 1), trend="ct")
    dgp = ARMApqWithTrendDGP(phis=[0.3], thetas=[0.1], delta=0.02, sigma=1.0, seed=SEED)
    verify_dgp("B.17 -- ARMA(1,1) rho=0.30 (delta=0.02)", dgp, {}, cl, CHECKS_ARMA)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.17")
    log("\n" + "="*60 + "\nB.17 -- ARMA(1,1) rho=0.30 (delta=0.02)\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(1, 0, 1)+trend")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="B.17 -- ARMA(1,1) rho=0.30 (delta=0.02)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA B.17 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # B.18 -- ARMA(1,1) rho=0.90  [tendencia leve delta=0.02]
    cl  = ARIMAWithTrendModel((1, 0, 1), trend="ct")
    dgp = ARMApqWithTrendDGP(phis=[0.9], thetas=[0.3], delta=0.02, sigma=1.0, seed=SEED)
    verify_dgp("B.18 -- ARMA(1,1) rho=0.90 (delta=0.02)", dgp, {}, cl, CHECKS_ARMA)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.18")
    log("\n" + "="*60 + "\nB.18 -- ARMA(1,1) rho=0.90 (delta=0.02)\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(1, 0, 1)+trend")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="B.18 -- ARMA(1,1) rho=0.90 (delta=0.02)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA B.18 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # B.19 -- ARMA(2,2) rho=0.30  [tendencia leve delta=0.02]
    cl  = ARIMAWithTrendModel((2, 0, 2), trend="ct")
    dgp = ARMApqWithTrendDGP(phis=[0.3, 0.1], thetas=[0.1, 0.05], delta=0.02, sigma=1.0, seed=SEED)
    verify_dgp("B.19 -- ARMA(2,2) rho=0.30 (delta=0.02)", dgp, {}, cl, CHECKS_ARMA)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.19")
    log("\n" + "="*60 + "\nB.19 -- ARMA(2,2) rho=0.30 (delta=0.02)\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(2, 0, 2)+trend")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="B.19 -- ARMA(2,2) rho=0.30 (delta=0.02)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA B.19 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # B.20 -- ARMA(2,2) rho=0.90  [tendencia leve delta=0.02]
    cl  = ARIMAWithTrendModel((2, 0, 2), trend="ct")
    dgp = ARMApqWithTrendDGP(phis=[0.9, -0.2], thetas=[0.3, -0.1], delta=0.02, sigma=1.0, seed=SEED)
    verify_dgp("B.20 -- ARMA(2,2) rho=0.90 (delta=0.02)", dgp, {}, cl, CHECKS_ARMA)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.20")
    log("\n" + "="*60 + "\nB.20 -- ARMA(2,2) rho=0.90 (delta=0.02)\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(2, 0, 2)+trend")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="B.20 -- ARMA(2,2) rho=0.90 (delta=0.02)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA B.20 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # B.21 -- ARMA(1,4) rho=0.30  [tendencia leve delta=0.02]
    cl  = ARIMAWithTrendModel((1, 0, 4), trend="ct")
    dgp = ARMApqWithTrendDGP(phis=[0.3], thetas=[0.1, 0.05, -0.03, 0.01], delta=0.02, sigma=1.0, seed=SEED)
    verify_dgp("B.21 -- ARMA(1,4) rho=0.30 (delta=0.02)", dgp, {}, cl, CHECKS_ARMA)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.21")
    log("\n" + "="*60 + "\nB.21 -- ARMA(1,4) rho=0.30 (delta=0.02)\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(1, 0, 4)+trend")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="B.21 -- ARMA(1,4) rho=0.30 (delta=0.02)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA B.21 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # B.22 -- ARMA(1,4) rho=0.90  [tendencia leve delta=0.02]
    cl  = ARIMAWithTrendModel((1, 0, 4), trend="ct")
    dgp = ARMApqWithTrendDGP(phis=[0.9], thetas=[0.3, -0.1, 0.05, -0.02], delta=0.02, sigma=1.0, seed=SEED)
    verify_dgp("B.22 -- ARMA(1,4) rho=0.90 (delta=0.02)", dgp, {}, cl, CHECKS_ARMA)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.22")
    log("\n" + "="*60 + "\nB.22 -- ARMA(1,4) rho=0.90 (delta=0.02)\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(1, 0, 4)+trend")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="B.22 -- ARMA(1,4) rho=0.90 (delta=0.02)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA B.22 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # B.23 -- ARMA(4,1) rho=0.30  [tendencia leve delta=0.02]
    cl  = ARIMAWithTrendModel((4, 0, 1), trend="ct")
    dgp = ARMApqWithTrendDGP(phis=[0.3, 0.1, 0.05, 0.02], thetas=[0.1], delta=0.02, sigma=1.0, seed=SEED)
    verify_dgp("B.23 -- ARMA(4,1) rho=0.30 (delta=0.02)", dgp, {}, cl, CHECKS_ARMA)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.23")
    log("\n" + "="*60 + "\nB.23 -- ARMA(4,1) rho=0.30 (delta=0.02)\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(4, 0, 1)+trend")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="B.23 -- ARMA(4,1) rho=0.30 (delta=0.02)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA B.23 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # B.24 -- ARMA(4,1) rho=0.90  [tendencia leve delta=0.02]
    cl  = ARIMAWithTrendModel((4, 0, 1), trend="ct")
    dgp = ARMApqWithTrendDGP(phis=[0.9, -0.2, 0.1, -0.05], thetas=[0.3], delta=0.02, sigma=1.0, seed=SEED)
    verify_dgp("B.24 -- ARMA(4,1) rho=0.90 (delta=0.02)", dgp, {}, cl, CHECKS_ARMA)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.24")
    log("\n" + "="*60 + "\nB.24 -- ARMA(4,1) rho=0.90 (delta=0.02)\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(4, 0, 1)+trend")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="B.24 -- ARMA(4,1) rho=0.90 (delta=0.02)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA B.24 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # B.25 -- AR(1) rho=0.30  [tendencia fuerte delta=0.1]
    cl  = ARIMAWithTrendModel((1, 0, 0), trend="ct")
    dgp = ARMApqWithTrendDGP(phis=[0.3], thetas=[], delta=0.1, sigma=1.0, seed=SEED)
    verify_dgp("B.25 -- AR(1) rho=0.30 (delta=0.1)", dgp, {}, cl, CHECKS_AR_TREND)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.25")
    log("\n" + "="*60 + "\nB.25 -- AR(1) rho=0.30 (delta=0.1)\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(1, 0, 0)+trend")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="B.25 -- AR(1) rho=0.30 (delta=0.1)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA B.25 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # B.26 -- AR(1) rho=0.90  [tendencia fuerte delta=0.1]
    cl  = ARIMAWithTrendModel((1, 0, 0), trend="ct")
    dgp = ARMApqWithTrendDGP(phis=[0.9], thetas=[], delta=0.1, sigma=1.0, seed=SEED)
    verify_dgp("B.26 -- AR(1) rho=0.90 (delta=0.1)", dgp, {}, cl, CHECKS_AR_TREND)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.26")
    log("\n" + "="*60 + "\nB.26 -- AR(1) rho=0.90 (delta=0.1)\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(1, 0, 0)+trend")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="B.26 -- AR(1) rho=0.90 (delta=0.1)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA B.26 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # B.27 -- AR(2) rho=0.30  [tendencia fuerte delta=0.1]
    cl  = ARIMAWithTrendModel((2, 0, 0), trend="ct")
    dgp = ARMApqWithTrendDGP(phis=[0.3, 0.1], thetas=[], delta=0.1, sigma=1.0, seed=SEED)
    verify_dgp("B.27 -- AR(2) rho=0.30 (delta=0.1)", dgp, {}, cl, CHECKS_AR_TREND)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.27")
    log("\n" + "="*60 + "\nB.27 -- AR(2) rho=0.30 (delta=0.1)\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(2, 0, 0)+trend")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="B.27 -- AR(2) rho=0.30 (delta=0.1)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA B.27 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # B.28 -- AR(2) rho=0.90  [tendencia fuerte delta=0.1]
    cl  = ARIMAWithTrendModel((2, 0, 0), trend="ct")
    dgp = ARMApqWithTrendDGP(phis=[0.9, -0.2], thetas=[], delta=0.1, sigma=1.0, seed=SEED)
    verify_dgp("B.28 -- AR(2) rho=0.90 (delta=0.1)", dgp, {}, cl, CHECKS_AR_TREND)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.28")
    log("\n" + "="*60 + "\nB.28 -- AR(2) rho=0.90 (delta=0.1)\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(2, 0, 0)+trend")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="B.28 -- AR(2) rho=0.90 (delta=0.1)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA B.28 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # B.29 -- AR(3) rho=0.30  [tendencia fuerte delta=0.1]
    cl  = ARIMAWithTrendModel((3, 0, 0), trend="ct")
    dgp = ARMApqWithTrendDGP(phis=[0.3, 0.1, 0.05], thetas=[], delta=0.1, sigma=1.0, seed=SEED)
    verify_dgp("B.29 -- AR(3) rho=0.30 (delta=0.1)", dgp, {}, cl, CHECKS_AR_TREND)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.29")
    log("\n" + "="*60 + "\nB.29 -- AR(3) rho=0.30 (delta=0.1)\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(3, 0, 0)+trend")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="B.29 -- AR(3) rho=0.30 (delta=0.1)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA B.29 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # B.30 -- AR(3) rho=0.90  [tendencia fuerte delta=0.1]
    cl  = ARIMAWithTrendModel((3, 0, 0), trend="ct")
    dgp = ARMApqWithTrendDGP(phis=[0.9, -0.2, 0.1], thetas=[], delta=0.1, sigma=1.0, seed=SEED)
    verify_dgp("B.30 -- AR(3) rho=0.90 (delta=0.1)", dgp, {}, cl, CHECKS_AR_TREND)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.30")
    log("\n" + "="*60 + "\nB.30 -- AR(3) rho=0.90 (delta=0.1)\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(3, 0, 0)+trend")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="B.30 -- AR(3) rho=0.90 (delta=0.1)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA B.30 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # B.31 -- AR(4) rho=0.30  [tendencia fuerte delta=0.1]
    cl  = ARIMAWithTrendModel((4, 0, 0), trend="ct")
    dgp = ARMApqWithTrendDGP(phis=[0.3, 0.1, 0.05, 0.02], thetas=[], delta=0.1, sigma=1.0, seed=SEED)
    verify_dgp("B.31 -- AR(4) rho=0.30 (delta=0.1)", dgp, {}, cl, CHECKS_AR_TREND)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.31")
    log("\n" + "="*60 + "\nB.31 -- AR(4) rho=0.30 (delta=0.1)\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(4, 0, 0)+trend")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="B.31 -- AR(4) rho=0.30 (delta=0.1)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA B.31 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # B.32 -- AR(4) rho=0.90  [tendencia fuerte delta=0.1]
    cl  = ARIMAWithTrendModel((4, 0, 0), trend="ct")
    dgp = ARMApqWithTrendDGP(phis=[0.9, -0.2, 0.1, -0.05], thetas=[], delta=0.1, sigma=1.0, seed=SEED)
    verify_dgp("B.32 -- AR(4) rho=0.90 (delta=0.1)", dgp, {}, cl, CHECKS_AR_TREND)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.32")
    log("\n" + "="*60 + "\nB.32 -- AR(4) rho=0.90 (delta=0.1)\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(4, 0, 0)+trend")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="B.32 -- AR(4) rho=0.90 (delta=0.1)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA B.32 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # B.33 -- MA(1) theta=0.30  [tendencia fuerte delta=0.1]
    cl  = ARIMAWithTrendModel((0, 0, 1), trend="ct")
    dgp = ARMApqWithTrendDGP(phis=[], thetas=[0.3], delta=0.1, sigma=1.0, seed=SEED)
    verify_dgp("B.33 -- MA(1) theta=0.30 (delta=0.1)", dgp, {}, cl, CHECKS_MA_TREND)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.33")
    log("\n" + "="*60 + "\nB.33 -- MA(1) theta=0.30 (delta=0.1)\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(0, 0, 1)+trend")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="B.33 -- MA(1) theta=0.30 (delta=0.1)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA B.33 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # B.34 -- MA(1) theta=0.90  [tendencia fuerte delta=0.1]
    cl  = ARIMAWithTrendModel((0, 0, 1), trend="ct")
    dgp = ARMApqWithTrendDGP(phis=[], thetas=[0.9], delta=0.1, sigma=1.0, seed=SEED)
    verify_dgp("B.34 -- MA(1) theta=0.90 (delta=0.1)", dgp, {}, cl, CHECKS_MA_TREND)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.34")
    log("\n" + "="*60 + "\nB.34 -- MA(1) theta=0.90 (delta=0.1)\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(0, 0, 1)+trend")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="B.34 -- MA(1) theta=0.90 (delta=0.1)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA B.34 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # B.35 -- MA(2) theta=0.30  [tendencia fuerte delta=0.1]
    cl  = ARIMAWithTrendModel((0, 0, 2), trend="ct")
    dgp = ARMApqWithTrendDGP(phis=[], thetas=[0.3, 0.1], delta=0.1, sigma=1.0, seed=SEED)
    verify_dgp("B.35 -- MA(2) theta=0.30 (delta=0.1)", dgp, {}, cl, CHECKS_MA_TREND)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.35")
    log("\n" + "="*60 + "\nB.35 -- MA(2) theta=0.30 (delta=0.1)\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(0, 0, 2)+trend")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="B.35 -- MA(2) theta=0.30 (delta=0.1)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA B.35 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # B.36 -- MA(2) theta=0.90  [tendencia fuerte delta=0.1]
    cl  = ARIMAWithTrendModel((0, 0, 2), trend="ct")
    dgp = ARMApqWithTrendDGP(phis=[], thetas=[0.9, 0.1], delta=0.1, sigma=1.0, seed=SEED)
    verify_dgp("B.36 -- MA(2) theta=0.90 (delta=0.1)", dgp, {}, cl, CHECKS_MA_TREND)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.36")
    log("\n" + "="*60 + "\nB.36 -- MA(2) theta=0.90 (delta=0.1)\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(0, 0, 2)+trend")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="B.36 -- MA(2) theta=0.90 (delta=0.1)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA B.36 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # B.37 -- MA(3) theta=0.30  [tendencia fuerte delta=0.1]
    cl  = ARIMAWithTrendModel((0, 0, 3), trend="ct")
    dgp = ARMApqWithTrendDGP(phis=[], thetas=[0.3, 0.1, -0.05], delta=0.1, sigma=1.0, seed=SEED)
    verify_dgp("B.37 -- MA(3) theta=0.30 (delta=0.1)", dgp, {}, cl, CHECKS_MA_TREND)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.37")
    log("\n" + "="*60 + "\nB.37 -- MA(3) theta=0.30 (delta=0.1)\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(0, 0, 3)+trend")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="B.37 -- MA(3) theta=0.30 (delta=0.1)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA B.37 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # B.38 -- MA(3) theta=0.90  [tendencia fuerte delta=0.1]
    cl  = ARIMAWithTrendModel((0, 0, 3), trend="ct")
    dgp = ARMApqWithTrendDGP(phis=[], thetas=[0.9, 0.1, -0.05], delta=0.1, sigma=1.0, seed=SEED)
    verify_dgp("B.38 -- MA(3) theta=0.90 (delta=0.1)", dgp, {}, cl, CHECKS_MA_TREND)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.38")
    log("\n" + "="*60 + "\nB.38 -- MA(3) theta=0.90 (delta=0.1)\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(0, 0, 3)+trend")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="B.38 -- MA(3) theta=0.90 (delta=0.1)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA B.38 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # B.39 -- MA(4) theta=0.30  [tendencia fuerte delta=0.1]
    cl  = ARIMAWithTrendModel((0, 0, 4), trend="ct")
    dgp = ARMApqWithTrendDGP(phis=[], thetas=[0.3, 0.1, -0.05, 0.02], delta=0.1, sigma=1.0, seed=SEED)
    verify_dgp("B.39 -- MA(4) theta=0.30 (delta=0.1)", dgp, {}, cl, CHECKS_MA_TREND)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.39")
    log("\n" + "="*60 + "\nB.39 -- MA(4) theta=0.30 (delta=0.1)\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(0, 0, 4)+trend")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="B.39 -- MA(4) theta=0.30 (delta=0.1)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA B.39 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # B.40 -- MA(4) theta=0.90  [tendencia fuerte delta=0.1]
    cl  = ARIMAWithTrendModel((0, 0, 4), trend="ct")
    dgp = ARMApqWithTrendDGP(phis=[], thetas=[0.9, 0.1, -0.05, 0.02], delta=0.1, sigma=1.0, seed=SEED)
    verify_dgp("B.40 -- MA(4) theta=0.90 (delta=0.1)", dgp, {}, cl, CHECKS_MA_TREND)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.40")
    log("\n" + "="*60 + "\nB.40 -- MA(4) theta=0.90 (delta=0.1)\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(0, 0, 4)+trend")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="B.40 -- MA(4) theta=0.90 (delta=0.1)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA B.40 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # B.41 -- ARMA(1,1) rho=0.30  [tendencia fuerte delta=0.1]
    cl  = ARIMAWithTrendModel((1, 0, 1), trend="ct")
    dgp = ARMApqWithTrendDGP(phis=[0.3], thetas=[0.1], delta=0.1, sigma=1.0, seed=SEED)
    verify_dgp("B.41 -- ARMA(1,1) rho=0.30 (delta=0.1)", dgp, {}, cl, CHECKS_ARMA)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.41")
    log("\n" + "="*60 + "\nB.41 -- ARMA(1,1) rho=0.30 (delta=0.1)\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(1, 0, 1)+trend")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="B.41 -- ARMA(1,1) rho=0.30 (delta=0.1)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA B.41 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # B.42 -- ARMA(1,1) rho=0.90  [tendencia fuerte delta=0.1]
    cl  = ARIMAWithTrendModel((1, 0, 1), trend="ct")
    dgp = ARMApqWithTrendDGP(phis=[0.9], thetas=[0.3], delta=0.1, sigma=1.0, seed=SEED)
    verify_dgp("B.42 -- ARMA(1,1) rho=0.90 (delta=0.1)", dgp, {}, cl, CHECKS_ARMA)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.42")
    log("\n" + "="*60 + "\nB.42 -- ARMA(1,1) rho=0.90 (delta=0.1)\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(1, 0, 1)+trend")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="B.42 -- ARMA(1,1) rho=0.90 (delta=0.1)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA B.42 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # B.43 -- ARMA(2,2) rho=0.30  [tendencia fuerte delta=0.1]
    cl  = ARIMAWithTrendModel((2, 0, 2), trend="ct")
    dgp = ARMApqWithTrendDGP(phis=[0.3, 0.1], thetas=[0.1, 0.05], delta=0.1, sigma=1.0, seed=SEED)
    verify_dgp("B.43 -- ARMA(2,2) rho=0.30 (delta=0.1)", dgp, {}, cl, CHECKS_ARMA)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.43")
    log("\n" + "="*60 + "\nB.43 -- ARMA(2,2) rho=0.30 (delta=0.1)\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(2, 0, 2)+trend")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="B.43 -- ARMA(2,2) rho=0.30 (delta=0.1)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA B.43 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # B.44 -- ARMA(2,2) rho=0.90  [tendencia fuerte delta=0.1]
    cl  = ARIMAWithTrendModel((2, 0, 2), trend="ct")
    dgp = ARMApqWithTrendDGP(phis=[0.9, -0.2], thetas=[0.3, -0.1], delta=0.1, sigma=1.0, seed=SEED)
    verify_dgp("B.44 -- ARMA(2,2) rho=0.90 (delta=0.1)", dgp, {}, cl, CHECKS_ARMA)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.44")
    log("\n" + "="*60 + "\nB.44 -- ARMA(2,2) rho=0.90 (delta=0.1)\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(2, 0, 2)+trend")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="B.44 -- ARMA(2,2) rho=0.90 (delta=0.1)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA B.44 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # B.45 -- ARMA(1,4) rho=0.30  [tendencia fuerte delta=0.1]
    cl  = ARIMAWithTrendModel((1, 0, 4), trend="ct")
    dgp = ARMApqWithTrendDGP(phis=[0.3], thetas=[0.1, 0.05, -0.03, 0.01], delta=0.1, sigma=1.0, seed=SEED)
    verify_dgp("B.45 -- ARMA(1,4) rho=0.30 (delta=0.1)", dgp, {}, cl, CHECKS_ARMA)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.45")
    log("\n" + "="*60 + "\nB.45 -- ARMA(1,4) rho=0.30 (delta=0.1)\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(1, 0, 4)+trend")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="B.45 -- ARMA(1,4) rho=0.30 (delta=0.1)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA B.45 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # B.46 -- ARMA(1,4) rho=0.90  [tendencia fuerte delta=0.1]
    cl  = ARIMAWithTrendModel((1, 0, 4), trend="ct")
    dgp = ARMApqWithTrendDGP(phis=[0.9], thetas=[0.3, -0.1, 0.05, -0.02], delta=0.1, sigma=1.0, seed=SEED)
    verify_dgp("B.46 -- ARMA(1,4) rho=0.90 (delta=0.1)", dgp, {}, cl, CHECKS_ARMA)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.46")
    log("\n" + "="*60 + "\nB.46 -- ARMA(1,4) rho=0.90 (delta=0.1)\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(1, 0, 4)+trend")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="B.46 -- ARMA(1,4) rho=0.90 (delta=0.1)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA B.46 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # B.47 -- ARMA(4,1) rho=0.30  [tendencia fuerte delta=0.1]
    cl  = ARIMAWithTrendModel((4, 0, 1), trend="ct")
    dgp = ARMApqWithTrendDGP(phis=[0.3, 0.1, 0.05, 0.02], thetas=[0.1], delta=0.1, sigma=1.0, seed=SEED)
    verify_dgp("B.47 -- ARMA(4,1) rho=0.30 (delta=0.1)", dgp, {}, cl, CHECKS_ARMA)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.47")
    log("\n" + "="*60 + "\nB.47 -- ARMA(4,1) rho=0.30 (delta=0.1)\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(4, 0, 1)+trend")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="B.47 -- ARMA(4,1) rho=0.30 (delta=0.1)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA B.47 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # B.48 -- ARMA(4,1) rho=0.90  [tendencia fuerte delta=0.1]
    cl  = ARIMAWithTrendModel((4, 0, 1), trend="ct")
    dgp = ARMApqWithTrendDGP(phis=[0.9, -0.2, 0.1, -0.05], thetas=[0.3], delta=0.1, sigma=1.0, seed=SEED)
    verify_dgp("B.48 -- ARMA(4,1) rho=0.90 (delta=0.1)", dgp, {}, cl, CHECKS_ARMA)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="B.48")
    log("\n" + "="*60 + "\nB.48 -- ARMA(4,1) rho=0.90 (delta=0.1)\n" + "="*60)
    build_grid_table(res, classical_name="ARIMA(4, 0, 1)+trend")
    plot_simulation_v3(dgp, [cl, chronos], {}, title="B.48 -- ARMA(4,1) rho=0.90 (delta=0.1)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA B.48 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

---
## Bloque C — Random Walk (3 experimentos)

DGP: `Y_t = drift + Y_{t-1} + eps_t`  
Modelo clasico: ARIMA(0,1,0)

In [ ]:
try:
    # C.1 -- RW sin drift
    cl  = ARIMAModel((0, 1, 0))
    dgp = RandomWalk(seed=SEED)
    verify_dgp("C.1 -- RW sin drift", dgp, {"drift": 0.0, "sigma": 1.0}, cl, CHECKS_RW)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {"drift": 0.0, "sigma": 1.0}, exp_id="C.1")
    log("\n" + "="*60 + "\nC.1 -- RW sin drift\n" + "="*60)
    build_grid_table(res, classical_name=cl.name)
    plot_simulation_v3(dgp, [cl, chronos], {"drift": 0.0, "sigma": 1.0}, title="C.1 -- Random Walk sin drift")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA C.1 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # C.2 -- RW drift leve (delta=0.05)
    cl  = ARIMAModel((0, 1, 0))
    dgp = RandomWalk(seed=SEED)
    verify_dgp("C.2 -- RW drift leve (delta=0.05)", dgp, {"drift": 0.05, "sigma": 1.0}, cl, CHECKS_RW)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {"drift": 0.05, "sigma": 1.0}, exp_id="C.2")
    log("\n" + "="*60 + "\nC.2 -- RW drift leve (delta=0.05)\n" + "="*60)
    build_grid_table(res, classical_name=cl.name)
    plot_simulation_v3(dgp, [cl, chronos], {"drift": 0.05, "sigma": 1.0}, title="C.2 -- Random Walk drift leve (delta=0.05)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA C.2 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # C.3 -- RW drift fuerte (delta=0.20)
    cl  = ARIMAModel((0, 1, 0))
    dgp = RandomWalk(seed=SEED)
    verify_dgp("C.3 -- RW drift fuerte (delta=0.20)", dgp, {"drift": 0.20, "sigma": 1.0}, cl, CHECKS_RW)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {"drift": 0.20, "sigma": 1.0}, exp_id="C.3")
    log("\n" + "="*60 + "\nC.3 -- RW drift fuerte (delta=0.20)\n" + "="*60)
    build_grid_table(res, classical_name=cl.name)
    plot_simulation_v3(dgp, [cl, chronos], {"drift": 0.20, "sigma": 1.0}, title="C.3 -- Random Walk drift fuerte (delta=0.20)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA C.3 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

---
## Bloque D — Volatilidad condicional: ARCH/GARCH (4 experimentos)

DGP: AR(1) en la media + proceso de varianza condicional  
Modelo clasico: AR+ARCH o AR+GARCH correctamente especificado

In [ ]:
try:
    # D.1 -- AR(1)-ARCH(1) leve  (alpha=0.10)
    cl  = ARARCHModel(ar_lags=1, p=1)
    dgp = AR1ARCH(seed=SEED)
    verify_dgp("D.1 -- AR(1)-ARCH(1) leve (alpha=0.10)", dgp, {"phi": 0.5, "omega": 0.5, "alpha": 0.10}, cl, CHECKS_ARCH)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos],
                  {"phi": 0.5, "omega": 0.5, "alpha": 0.10}, exp_id="D.1")
    log("\n" + "="*60 + "\nD.1 -- AR(1)-ARCH(1) leve\n" + "="*60)
    build_grid_table(res, classical_name=cl.name)
    plot_simulation_v3(dgp, [cl, chronos], {"phi": 0.5, "omega": 0.5, "alpha": 0.10}, title="D.1 -- AR(1)-ARCH(1) leve (alpha=0.10)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA D.1 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # D.2 -- AR(1)-ARCH(1) fuerte  (alpha=0.50)
    cl  = ARARCHModel(ar_lags=1, p=1)
    dgp = AR1ARCH(seed=SEED)
    verify_dgp("D.2 -- AR(1)-ARCH(1) fuerte (alpha=0.50)", dgp, {"phi": 0.5, "omega": 0.5, "alpha": 0.50}, cl, CHECKS_ARCH)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos],
                  {"phi": 0.5, "omega": 0.5, "alpha": 0.50}, exp_id="D.2")
    log("\n" + "="*60 + "\nD.2 -- AR(1)-ARCH(1) fuerte\n" + "="*60)
    build_grid_table(res, classical_name=cl.name)
    plot_simulation_v3(dgp, [cl, chronos], {"phi": 0.5, "omega": 0.5, "alpha": 0.50}, title="D.2 -- AR(1)-ARCH(1) fuerte (alpha=0.50)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA D.2 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # D.3 -- AR(1)-GARCH(1,1) baja persistencia  (alpha+beta=0.50)
    cl  = ARGARCHModel(ar_lags=1, p=1, q=1)
    dgp = AR1GARCH(seed=SEED)
    verify_dgp("D.3 -- AR(1)-GARCH(1,1) baja persistencia", dgp, {"phi": 0.5, "omega": 0.5, "alpha": 0.10, "beta": 0.40}, cl, CHECKS_ARCH)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos],
                  {"phi": 0.5, "omega": 0.5, "alpha": 0.10, "beta": 0.40}, exp_id="D.3")
    log("\n" + "="*60 + "\nD.3 -- AR(1)-GARCH(1,1) baja persistencia\n" + "="*60)
    build_grid_table(res, classical_name=cl.name)
    plot_simulation_v3(dgp, [cl, chronos], {"phi": 0.5, "omega": 0.5, "alpha": 0.10, "beta": 0.40}, title="D.3 -- AR(1)-GARCH(1,1) baja persistencia (alpha+beta=0.50)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA D.3 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # D.4 -- AR(1)-GARCH(1,1) alta persistencia  (alpha+beta=0.95)
    cl  = ARGARCHModel(ar_lags=1, p=1, q=1)
    dgp = AR1GARCH(seed=SEED)
    verify_dgp("D.4 -- AR(1)-GARCH(1,1) alta persistencia", dgp, {"phi": 0.5, "omega": 0.1, "alpha": 0.10, "beta": 0.85}, cl, CHECKS_ARCH)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos],
                  {"phi": 0.5, "omega": 0.1, "alpha": 0.10, "beta": 0.85}, exp_id="D.4")
    log("\n" + "="*60 + "\nD.4 -- AR(1)-GARCH(1,1) alta persistencia\n" + "="*60)
    build_grid_table(res, classical_name=cl.name)
    plot_simulation_v3(dgp, [cl, chronos], {"phi": 0.5, "omega": 0.1, "alpha": 0.10, "beta": 0.85}, title="D.4 -- AR(1)-GARCH(1,1) alta persistencia (alpha+beta=0.95)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA D.4 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

---
## Bloque E — ETS y Theta (8 experimentos)

DGPs de espacio de estados: nivel local, tendencia, estacionalidad  
Modelos clasicos: ETS(A,T,S) y Theta

In [ ]:
try:
    # E.1 -- Local Level  ETS(A,N,N)
    cl  = ETSModel()
    dgp = LocalLevelDGP(seed=SEED)
    verify_dgp("E.1 -- Local Level ETS(A,N,N)", dgp, {"sigma_eps": 1.0, "sigma_eta": 0.10}, cl, CHECKS_ETS)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos],
                  {"sigma_eps": 1.0, "sigma_eta": 0.10}, exp_id="E.1")
    log("\n" + "="*60 + "\nE.1 -- Local Level ETS(A,N,N)\n" + "="*60)
    build_grid_table(res, classical_name=cl.name)
    plot_simulation_v3(dgp, [cl, chronos], {"sigma_eps": 1.0, "sigma_eta": 0.10}, title="E.1 -- Local Level ETS(A,N,N)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA E.1 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # E.2 -- Local Linear Trend leve  ETS(A,A,N)  sigma_z=0.05
    cl  = ETSModel(trend="add")
    dgp = LocalTrendDGP(seed=SEED)
    verify_dgp("E.2 -- LLT leve ETS(A,A,N)", dgp, {"sigma_eps": 1.0, "sigma_eta": 0.1, "sigma_zeta": 0.05, "b0": 0.1}, cl, CHECKS_ETS)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos],
                  {"sigma_eps": 1.0, "sigma_eta": 0.1, "sigma_zeta": 0.05, "b0": 0.1}, exp_id="E.2")
    log("\n" + "="*60 + "\nE.2 -- LLT leve ETS(A,A,N)\n" + "="*60)
    build_grid_table(res, classical_name=cl.name)
    plot_simulation_v3(dgp, [cl, chronos], {"sigma_eps": 1.0, "sigma_eta": 0.1, "sigma_zeta": 0.05, "b0": 0.1}, title="E.2 -- Local Linear Trend leve (sigma_z=0.05)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA E.2 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # E.3 -- Local Linear Trend fuerte  ETS(A,A,N)  sigma_z=0.20
    cl  = ETSModel(trend="add")
    dgp = LocalTrendDGP(seed=SEED)
    verify_dgp("E.3 -- LLT fuerte ETS(A,A,N)", dgp, {"sigma_eps": 1.0, "sigma_eta": 0.2, "sigma_zeta": 0.20, "b0": 0.5}, cl, CHECKS_ETS)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos],
                  {"sigma_eps": 1.0, "sigma_eta": 0.2, "sigma_zeta": 0.20, "b0": 0.5}, exp_id="E.3")
    log("\n" + "="*60 + "\nE.3 -- LLT fuerte ETS(A,A,N)\n" + "="*60)
    build_grid_table(res, classical_name=cl.name)
    plot_simulation_v3(dgp, [cl, chronos], {"sigma_eps": 1.0, "sigma_eta": 0.2, "sigma_zeta": 0.20, "b0": 0.5}, title="E.3 -- Local Linear Trend fuerte (sigma_z=0.20)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA E.3 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # E.4 -- Damped Trend  ETS(A,Ad,N)  phi_d=0.90
    cl  = ETSModel(trend="add", damped_trend=True)
    dgp = DampedTrendDGP(seed=SEED)
    verify_dgp("E.4 -- Damped Trend ETS(A,Ad,N)", dgp, {"phi": 0.9, "sigma_eps": 1.0, "sigma_eta": 0.2, "sigma_zeta": 0.1}, cl, CHECKS_ETS)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos],
                  {"phi": 0.9, "sigma_eps": 1.0, "sigma_eta": 0.2, "sigma_zeta": 0.1}, exp_id="E.4")
    log("\n" + "="*60 + "\nE.4 -- Damped Trend ETS(A,Ad,N)\n" + "="*60)
    build_grid_table(res, classical_name=cl.name)
    plot_simulation_v3(dgp, [cl, chronos], {"phi": 0.9, "sigma_eps": 1.0, "sigma_eta": 0.2, "sigma_zeta": 0.1}, title="E.4 -- Damped Trend ETS(A,Ad,N)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA E.4 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # E.5 -- Seasonal Aditiva s=12  ETS(A,N,A)
    cl  = ETSModel(seasonal="add", seasonal_periods=12)
    dgp = LocalLevelSeasonalDGP(seed=SEED)
    verify_dgp("E.5 -- Seasonal Aditiva s=12", dgp,
               {"s": 12, "sigma_eps": 0.5, "sigma_eta": 0.1, "sigma_zeta": 0.0, "sigma_omega": 0.05, "b0": 0.0},
               cl, CHECKS_ETS)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos],
                  {"s": 12, "sigma_eps": 0.5, "sigma_eta": 0.1,
                   "sigma_zeta": 0.0, "sigma_omega": 0.05, "b0": 0.0},
                  exp_id="E.5", T_list=[50, 100, 200])
    log("\n" + "="*60 + "\nE.5 -- Seasonal Aditiva s=12 ETS(A,N,A)\n" + "="*60)
    build_grid_table(res, classical_name=cl.name)
    plot_simulation_v3(dgp, [cl, chronos], {"s": 12, "sigma_eps": 0.5, "sigma_eta": 0.1, "sigma_zeta": 0.0, "sigma_omega": 0.05, "b0": 0.0}, title="E.5 -- Seasonal Aditiva s=12 ETS(A,N,A)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA E.5 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # E.6 -- Trend + Seasonal s=12  ETS(A,A,A)
    cl  = ETSModel(trend="add", seasonal="add", seasonal_periods=12)
    dgp = LocalLevelSeasonalDGP(seed=SEED)
    verify_dgp("E.6 -- Trend+Seasonal s=12 ETS(A,A,A)", dgp,
               {"s": 12, "sigma_eps": 0.5, "sigma_eta": 0.1, "sigma_zeta": 0.05, "sigma_omega": 0.05, "b0": 0.1},
               cl, CHECKS_ETS)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos],
                  {"s": 12, "sigma_eps": 0.5, "sigma_eta": 0.1,
                   "sigma_zeta": 0.05, "sigma_omega": 0.05, "b0": 0.1},
                  exp_id="E.6", T_list=[50, 100, 200])
    log("\n" + "="*60 + "\nE.6 -- Trend+Seasonal s=12 ETS(A,A,A)\n" + "="*60)
    build_grid_table(res, classical_name=cl.name)
    plot_simulation_v3(dgp, [cl, chronos], {"s": 12, "sigma_eps": 0.5, "sigma_eta": 0.1, "sigma_zeta": 0.05, "sigma_omega": 0.05, "b0": 0.1}, title="E.6 -- Trend+Seasonal s=12 ETS(A,A,A)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA E.6 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # E.7 -- Theta leve  (b0=0.10, sigma_z=0.01)
    cl  = ThetaModel()
    dgp = LocalTrendDGP(seed=SEED)
    verify_dgp("E.7 -- Theta leve", dgp, {"sigma_eps": 1.0, "sigma_eta": 0.1, "sigma_zeta": 0.01, "b0": 0.10}, cl, CHECKS_THETA)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos],
                  {"sigma_eps": 1.0, "sigma_eta": 0.1, "sigma_zeta": 0.01, "b0": 0.10}, exp_id="E.7")
    log("\n" + "="*60 + "\nE.7 -- Theta leve\n" + "="*60)
    build_grid_table(res, classical_name=cl.name)
    plot_simulation_v3(dgp, [cl, chronos], {"sigma_eps": 1.0, "sigma_eta": 0.1, "sigma_zeta": 0.01, "b0": 0.10}, title="E.7 -- Theta leve (b0=0.10, sigma_z=0.01)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA E.7 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # E.8 -- Theta fuerte  (b0=0.50, sigma_z=0.10)
    cl  = ThetaModel()
    dgp = LocalTrendDGP(seed=SEED)
    verify_dgp("E.8 -- Theta fuerte", dgp, {"sigma_eps": 1.0, "sigma_eta": 0.2, "sigma_zeta": 0.10, "b0": 0.50}, cl, CHECKS_THETA)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos],
                  {"sigma_eps": 1.0, "sigma_eta": 0.2, "sigma_zeta": 0.10, "b0": 0.50}, exp_id="E.8")
    log("\n" + "="*60 + "\nE.8 -- Theta fuerte\n" + "="*60)
    build_grid_table(res, classical_name=cl.name)
    plot_simulation_v3(dgp, [cl, chronos], {"sigma_eps": 1.0, "sigma_eta": 0.2, "sigma_zeta": 0.10, "b0": 0.50}, title="E.8 -- Theta fuerte (b0=0.50, sigma_z=0.10)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA E.8 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

---
## Bloque F — SARIMA (6 experimentos)

DGP: SAR(1)(1)_s estacionario * (1-L)(1-L^s) integrado  
Periodos estacionales: s=4 (trimestral) * s=12 (mensual)

In [ ]:
try:
    # F.1 -- SAR(1)(1)_4 baja persist.
    cl  = SARIMAModel(order=(1, 0, 0), seasonal_order=(1, 0, 0, 4))
    dgp = SeasonalDGP(seed=SEED)
    verify_dgp("F.1 -- SAR(1)(1)_4 baja persist.", dgp, {"phi": 0.3, "Phi": 0.3, "s": 4, "sigma": 1.0, "integrated": False}, cl, CHECKS_SAR)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos],
                  {"phi": 0.3, "Phi": 0.3, "s": 4, "sigma": 1.0, "integrated": False},
                  exp_id="F.1", T_list=T_LIST)
    log("\n" + "="*60 + "\nF.1 -- SAR(1)(1)_4 baja persist.\n" + "="*60)
    build_grid_table(res, classical_name=cl.name)
    plot_simulation_v3(dgp, [cl, chronos], {"phi": 0.3, "Phi": 0.3, "s": 4, "sigma": 1.0, "integrated": False}, title="F.1 -- SAR(1)(1)_4 baja persist.")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA F.1 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # F.2 -- SAR(1)(1)_4 alta persist.
    cl  = SARIMAModel(order=(1, 0, 0), seasonal_order=(1, 0, 0, 4))
    dgp = SeasonalDGP(seed=SEED)
    verify_dgp("F.2 -- SAR(1)(1)_4 alta persist.", dgp, {"phi": 0.9, "Phi": 0.6, "s": 4, "sigma": 1.0, "integrated": False}, cl, CHECKS_SAR)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos],
                  {"phi": 0.9, "Phi": 0.6, "s": 4, "sigma": 1.0, "integrated": False},
                  exp_id="F.2", T_list=T_LIST)
    log("\n" + "="*60 + "\nF.2 -- SAR(1)(1)_4 alta persist.\n" + "="*60)
    build_grid_table(res, classical_name=cl.name)
    plot_simulation_v3(dgp, [cl, chronos], {"phi": 0.9, "Phi": 0.6, "s": 4, "sigma": 1.0, "integrated": False}, title="F.2 -- SAR(1)(1)_4 alta persist.")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA F.2 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # F.3 -- SAR(1)(1)_12 baja persist.
    cl  = SARIMAModel(order=(1, 0, 0), seasonal_order=(1, 0, 0, 12))
    dgp = SeasonalDGP(seed=SEED)
    verify_dgp("F.3 -- SAR(1)(1)_12 baja persist.", dgp, {"phi": 0.3, "Phi": 0.3, "s": 12, "sigma": 1.0, "integrated": False}, cl, CHECKS_SAR)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos],
                  {"phi": 0.3, "Phi": 0.3, "s": 12, "sigma": 1.0, "integrated": False},
                  exp_id="F.3", T_list=[50, 100, 200])
    log("\n" + "="*60 + "\nF.3 -- SAR(1)(1)_12 baja persist.\n" + "="*60)
    build_grid_table(res, classical_name=cl.name)
    plot_simulation_v3(dgp, [cl, chronos], {"phi": 0.3, "Phi": 0.3, "s": 12, "sigma": 1.0, "integrated": False}, title="F.3 -- SAR(1)(1)_12 baja persist.")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA F.3 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # F.4 -- SAR(1)(1)_12 alta persist.
    cl  = SARIMAModel(order=(1, 0, 0), seasonal_order=(1, 0, 0, 12))
    dgp = SeasonalDGP(seed=SEED)
    verify_dgp("F.4 -- SAR(1)(1)_12 alta persist.", dgp, {"phi": 0.9, "Phi": 0.6, "s": 12, "sigma": 1.0, "integrated": False}, cl, CHECKS_SAR)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos],
                  {"phi": 0.9, "Phi": 0.6, "s": 12, "sigma": 1.0, "integrated": False},
                  exp_id="F.4", T_list=[50, 100, 200])
    log("\n" + "="*60 + "\nF.4 -- SAR(1)(1)_12 alta persist.\n" + "="*60)
    build_grid_table(res, classical_name=cl.name)
    plot_simulation_v3(dgp, [cl, chronos], {"phi": 0.9, "Phi": 0.6, "s": 12, "sigma": 1.0, "integrated": False}, title="F.4 -- SAR(1)(1)_12 alta persist.")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA F.4 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # F.5 -- (1-L)(1-L^4) integrado
    cl  = SARIMAModel(order=(0, 1, 0), seasonal_order=(0, 1, 0, 4))
    dgp = SeasonalDGP(seed=SEED)
    verify_dgp("F.5 -- (1-L)(1-L^4) integrado", dgp, {"s": 4, "sigma": 1.0, "integrated": True}, cl, CHECKS_SAR)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos],
                  {"s": 4, "sigma": 1.0, "integrated": True},
                  exp_id="F.5", T_list=T_LIST)
    log("\n" + "="*60 + "\nF.5 -- (1-L)(1-L^4) integrado\n" + "="*60)
    build_grid_table(res, classical_name=cl.name)
    plot_simulation_v3(dgp, [cl, chronos], {"s": 4, "sigma": 1.0, "integrated": True}, title="F.5 -- (1-L)(1-L^4) integrado")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA F.5 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # F.6 -- (1-L)(1-L^12) integrado
    cl  = SARIMAModel(order=(0, 1, 0), seasonal_order=(0, 1, 0, 12))
    dgp = SeasonalDGP(seed=SEED)
    verify_dgp("F.6 -- (1-L)(1-L^12) integrado", dgp, {"s": 12, "sigma": 1.0, "integrated": True}, cl, CHECKS_SAR)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos],
                  {"s": 12, "sigma": 1.0, "integrated": True},
                  exp_id="F.6", T_list=[50, 100, 200])
    log("\n" + "="*60 + "\nF.6 -- (1-L)(1-L^12) integrado\n" + "="*60)
    build_grid_table(res, classical_name=cl.name)
    plot_simulation_v3(dgp, [cl, chronos], {"s": 12, "sigma": 1.0, "integrated": True}, title="F.6 -- (1-L)(1-L^12) integrado")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA F.6 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

---
## Bloque G — No linealidad de umbral: SETAR / LSTAR / ESTAR (4 experimentos)

Benchmark clasico: ARIMA(1,0,0) — lineal misspecificado  
Pregunta: detecta Chronos-2 la no-linealidad que ARIMA no puede capturar?  
- **SETAR**: umbral deterministico y observable  
- **LSTAR**: transicion suave asimetrica (ciclos de negocios)  
- **ESTAR**: transicion suave simetrica (tipos de cambio con bandas)

In [ ]:
try:
    # G.1 -- SETAR(2;1) baja persistencia  phi1=0.30, phi2=-0.30, k=0
    cl  = ARIMAModel((1, 0, 0))
    dgp = SETARDGp(phi1=0.30, phi2=-0.30, threshold=0.0, delay=1, sigma=1.0, seed=SEED)
    verify_dgp("G.1 -- SETAR(2;1) baja persistencia", dgp, {}, cl, CHECKS_NONLINEAR)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="G.1")
    log("\n" + "="*60 + "\nG.1 -- SETAR(2;1) baja persistencia\n" + "="*60)
    build_grid_table(res, classical_name=cl.name)
    plot_simulation_v3(dgp, [cl, chronos], {}, title="G.1 -- SETAR(2;1) phi1=0.30 / phi2=-0.30")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA G.1 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # G.2 -- SETAR(2;1) alta persistencia  phi1=0.90, phi2=-0.50, k=0
    cl  = ARIMAModel((1, 0, 0))
    dgp = SETARDGp(phi1=0.90, phi2=-0.50, threshold=0.0, delay=1, sigma=1.0, seed=SEED)
    verify_dgp("G.2 -- SETAR(2;1) alta persistencia", dgp, {}, cl, CHECKS_NONLINEAR)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="G.2")
    log("\n" + "="*60 + "\nG.2 -- SETAR(2;1) alta persistencia\n" + "="*60)
    build_grid_table(res, classical_name=cl.name)
    plot_simulation_v3(dgp, [cl, chronos], {}, title="G.2 -- SETAR(2;1) phi1=0.90 / phi2=-0.50")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA G.2 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # G.3 -- LSTAR(1) asimetrico  phi1=0.30, phi2=0.90, gamma=2, c=0
    cl  = ARIMAModel((1, 0, 0))
    dgp = LSTARDGp(phi1=0.30, phi2=0.90, gamma=2.0, c=0.0, delay=1, sigma=1.0, seed=SEED)
    verify_dgp("G.3 -- LSTAR(1) asimetrico", dgp, {}, cl, CHECKS_NONLINEAR)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="G.3")
    log("\n" + "="*60 + "\nG.3 -- LSTAR(1) asimetrico\n" + "="*60)
    build_grid_table(res, classical_name=cl.name)
    plot_simulation_v3(dgp, [cl, chronos], {}, title="G.3 -- LSTAR(1) phi1=0.30 / phi2=0.90 (gamma=2)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA G.3 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

In [ ]:
try:
    # G.4 -- ESTAR(1) simetrico  phi1=0.90, phi2=0.10, gamma=1, c=0
    cl  = ARIMAModel((1, 0, 0))
    dgp = ESTARDGp(phi1=0.90, phi2=0.10, gamma=1.0, c=0.0, delay=1, sigma=1.0, seed=SEED)
    verify_dgp("G.4 -- ESTAR(1) simetrico", dgp, {}, cl, CHECKS_NONLINEAR)
    res = run_exp(dgp, lambda T, _cl=cl: [_cl, chronos], {}, exp_id="G.4")
    log("\n" + "="*60 + "\nG.4 -- ESTAR(1) simetrico\n" + "="*60)
    build_grid_table(res, classical_name=cl.name)
    plot_simulation_v3(dgp, [cl, chronos], {}, title="G.4 -- ESTAR(1) phi1=0.90 / phi2=0.10 (gamma=1)")
except Exception as _exc:
    log("\n" + "!"*60)
    log("[CELDA G.4 FALLO] " + type(_exc).__name__ + ": " + str(_exc))
    log("!"*60)
    log(traceback.format_exc())

---
## Resumen

| Bloque | DGPs | T | R | Total series |
|--------|------|---|---|-------------|
| A — ARMA sin tendencia | 24 | 4 | 500 | 48,000 |
| B — ARMA con tendencia | 48 | 4 | 500 | 96,000 |
| C — Random Walk | 3 | 4 | 500 | 6,000 |
| D — ARCH/GARCH | 4 | 4 | 500 | 8,000 |
| E — ETS/Theta | 6* | 4 | 500 | ~12,000 |
| F — SARIMA | 6* | <=4 | 500 | ~10,000 |
| G — SETAR/LSTAR/ESTAR | 4 | 4 | 500 | 8,000 |
| **Total** | **95** | — | — | **~188,000** |

*Algunos experimentos usan T_list reducido (s=12 requiere T>=50)